In [1]:
import os
import glob
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import random
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import pprint

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 120)


# Load path and dataset

In [2]:
REPO_ROOT = Path.cwd()
DATA_DIR = REPO_ROOT / "data"

paths = {
    "lms": DATA_DIR / "lms_loan_daily.csv",
    "clickstream": DATA_DIR / "feature_clickstream.csv",
    "attributes": DATA_DIR / "features_attributes.csv",
    "financials": DATA_DIR / "features_financials.csv",
}

for name, p in paths.items():
    if not p.exists():
        raise FileNotFoundError(f"Missing {name}: {p}")

lms = pd.read_csv(paths["lms"])
click = pd.read_csv(paths["clickstream"])
attr = pd.read_csv(paths["attributes"])
fin = pd.read_csv(paths["financials"])

print({k: len(v) for k, v in [("lms", lms), ("clickstream", click), ("attributes", attr), ("financials", fin)]})

{'lms': 137500, 'clickstream': 215376, 'attributes': 12500, 'financials': 12500}


# lms_loan_daily.csv


In [3]:
for c in ["loan_start_date", "snapshot_date"]:
    lms[c] = pd.to_datetime(lms[c], errors="coerce")

n_loans = lms["loan_id"].nunique()
n_cust = lms["Customer_ID"].nunique()
rows_per_loan = lms.groupby("loan_id").size()

print("rows", len(lms))
print()
print("rows per loan_id")
print(rows_per_loan.describe())

rows 137500

rows per loan_id
count    12500.0
mean        11.0
std          0.0
min         11.0
25%         11.0
50%         11.0
75%         11.0
max         11.0
dtype: float64


In [4]:
print("loan_id nunique", n_loans)
print("Customer_ID nunique", n_cust)
print("rows", len(lms))
print("n_loans * 11:", n_loans * 11)
inst = sorted(lms["installment_num"].dropna().unique().tolist())
print("installment_num nunique", len(inst), inst)


loan_id nunique 12500
Customer_ID nunique 12500
rows 137500
n_loans * 11: 137500
installment_num nunique 11 [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


In [5]:
for col in ["tenure", "loan_amt"]:
    nu = lms[col].nunique()
    print(col, "nunique", nu, lms[col].dropna().unique()[:8])

tenure nunique 1 [10]
loan_amt nunique 1 [10000]


### tenure and loan_amt are two constant values across all table

In [6]:
print("loan_start_date total distinct month values (M):", lms["loan_start_date"].dt.to_period("M").nunique())
print("earliest and latest loan_start_date ", lms["loan_start_date"].min(), lms["loan_start_date"].max())
print("earliest and latest snapshot_date ", lms["snapshot_date"].min(), lms["snapshot_date"].max())


loan_start_date total distinct month values (M): 25
earliest and latest loan_start_date  2023-01-01 00:00:00 2025-01-01 00:00:00
earliest and latest snapshot_date  2023-01-01 00:00:00 2025-11-01 00:00:00


In [7]:
for col in ["due_amt", "paid_amt", "overdue_amt", "balance"]:
    print(col)
    print(lms[col].describe(percentiles=[0.5, 0.9, 0.99]))

pos = (lms["overdue_amt"] > 0).sum()
pct = pos / len(lms)
print("overdue_amt > 0", int(pos), f"{pct:.3%}")

due_amt
count    137500.000000
mean        909.090909
std         287.480833
min           0.000000
50%        1000.000000
90%        1000.000000
99%        1000.000000
max        1000.000000
Name: due_amt, dtype: float64
paid_amt
count    137500.000000
mean        711.810909
std         485.726859
min           0.000000
50%        1000.000000
90%        1000.000000
99%        2000.000000
max        4000.000000
Name: paid_amt, dtype: float64
overdue_amt
count    137500.000000
mean        871.905455
std        2002.672544
min           0.000000
50%           0.000000
90%        4000.000000
99%        8000.000000
max       10000.000000
Name: overdue_amt, dtype: float64
balance
count    137500.000000
mean       5871.905455
std        3070.777575
min           0.000000
50%        7000.000000
90%       10000.000000
99%       10000.000000
max       10000.000000
Name: balance, dtype: float64
overdue_amt > 0 28876 21.001%


In [8]:
print("due_amt by installment_num")
print(lms.groupby("installment_num")["due_amt"].agg(["mean", "min", "max"]))

print(lms.isna().sum().sort_values(ascending=False).head(10))
display(lms.head(5))


due_amt by installment_num
                   mean     min     max
installment_num                        
0                   0.0     0.0     0.0
1                1000.0  1000.0  1000.0
2                1000.0  1000.0  1000.0
3                1000.0  1000.0  1000.0
4                1000.0  1000.0  1000.0
5                1000.0  1000.0  1000.0
6                1000.0  1000.0  1000.0
7                1000.0  1000.0  1000.0
8                1000.0  1000.0  1000.0
9                1000.0  1000.0  1000.0
10               1000.0  1000.0  1000.0
loan_id            0
Customer_ID        0
loan_start_date    0
tenure             0
installment_num    0
loan_amt           0
due_amt            0
paid_amt           0
overdue_amt        0
balance            0
dtype: int64


,loan_id,Customer_ID,loan_start_date,tenure,installment_num,loan_amt,due_amt,paid_amt,overdue_amt,balance,snapshot_date
0,CUS_0x1000_2023_05_01,CUS_0x1000,2023-05-01,10,0,10000,0.0,0.0,0.0,10000.0,2023-05-01
1,CUS_0x1000_2023_05_01,CUS_0x1000,2023-05-01,10,1,10000,1000.0,1000.0,0.0,9000.0,2023-06-01
2,CUS_0x1000_2023_05_01,CUS_0x1000,2023-05-01,10,2,10000,1000.0,1000.0,0.0,8000.0,2023-07-01
3,CUS_0x1000_2023_05_01,CUS_0x1000,2023-05-01,10,3,10000,1000.0,0.0,1000.0,8000.0,2023-08-01
4,CUS_0x1000_2023_05_01,CUS_0x1000,2023-05-01,10,4,10000,1000.0,2000.0,0.0,6000.0,2023-09-01
